In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

In [4]:
LOCAL_DIR = Path("../local")
DATA_DIR = LOCAL_DIR / "dataset"
X_TRAIN_PATH = DATA_DIR / "x_train.csv"
Y_TRAIN_PATH = DATA_DIR / "y_train.csv"
X_TEST_PATH = DATA_DIR / "x_test.csv"

# Load and prepare data

In [5]:
x_train_df = pd.read_csv(X_TRAIN_PATH, engine="python")
y_train_df = pd.read_csv(Y_TRAIN_PATH, engine="python")
x_test_df = pd.read_csv(X_TEST_PATH, engine="python")

In [6]:
x_tr_np = x_train_df.to_numpy()
x_te_np = x_test_df.to_numpy()
x_np = np.concat([x_tr_np, x_te_np], axis=0)

n_trains = x_tr_np.shape[0]
n_tests = x_te_np.shape[0]

y_tr_np = y_train_df.to_numpy()

In [5]:
_, class_counts = np.unique(y_tr_np[:, 1], return_counts=True)

In [6]:
class_ratio = class_counts / class_counts.sum()

## Data helpers

In [7]:
def split_data(X: np.ndarray, y: np.ndarray, tr_rate: float = 0.8):
    np.random.seed(12345)
    
    y_flat = y.flatten()
    pos_ids = np.where(y_flat > 0)[0]
    neg_ids = np.where(y_flat < 0)[0]

    pos_perm = np.random.permutation(pos_ids)
    neg_perm = np.random.permutation(neg_ids)
    n_pos_tr = int(tr_rate * len(pos_perm))
    n_neg_tr = int(tr_rate * len(neg_perm))
    
    tr_ids = np.concatenate([pos_perm[:n_pos_tr], neg_perm[:n_neg_tr]])
    va_ids = np.concatenate([pos_perm[n_pos_tr:], neg_perm[n_neg_tr:]])

    return (X[tr_ids], y[tr_ids]), (X[va_ids], y[va_ids])

In [8]:
def k_fold(X: np.ndarray, y: np.ndarray, K: int = 5):
    np.random.seed(12345)
    N = y.shape[0]
    
    y_flat = y.flatten()
    pos_ids = np.where(y_flat > 0)[0]
    neg_ids = np.where(y_flat < 0)[0]

    pos_perm = np.random.permutation(pos_ids)
    neg_perm = np.random.permutation(neg_ids)
    n_pos_tr = len(pos_perm) // N
    n_neg_tr = len(neg_perm) // N
    
    fold_ids = [np.concatenate([pos_perm[k * n_pos_tr:(k+1) * n_pos_tr], neg_perm[k * n_neg_tr:(k+1) * n_neg_tr]]) for k in range(K)]
    return fold_ids

# Training

In [9]:
def acc(X: np.ndarray, y: np.ndarray, w: np.ndarray):
    return np.mean((X @ w) * y > 0)

In [10]:
def f1(X: np.ndarray, y: np.ndarray, w: np.ndarray):
    pred = (X @ w).flatten()
    y = y.flatten()
    tp = np.sum((pred >= 0) & (y > 0))
    fp = np.sum((pred >= 0) & (y < 0))
    fn = np.sum((pred < 0) & (y > 0))
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    return 2 * precision * recall / (precision + recall + 1e-8)

In [11]:
def sigmoid(x: np.ndarray):
    x = np.clip(x, -400, 400)
    return 1.0 / (1.0 + np.exp(-x))

In [12]:
def least_squares(X, y):
    return np.linalg.solve(X.T @ X + 1 * np.eye(X.shape[1]), X.T @ y)

In [13]:
def compute_loss(X: np.ndarray, y: np.ndarray, w: np.ndarray, loss: str = "logistic", lambda_: float = 0):
    if loss == "logistic":
        return np.mean(np.logaddexp(0, -y * (X @ w)))
    elif loss == "hinge":
        return np.mean(np.maximum(1 - y * (X @ w), 0))
    else:
        raise ValueError(f"{loss} is invalid")

In [14]:
def compute_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray, loss: str = "logistic", lambda_: float = 0):
    if loss == "logistic":
        return -np.expand_dims(np.mean(sigmoid(-y * (X @ w)) * y * X, axis=0), -1) + lambda_ * w
    elif loss == "hinge":
        return np.expand_dims(np.mean((-X * y) * (1 - y * (X @ w) > 0), axis=0), -1) + lambda_ * w
    else:
        raise ValueError(f"{loss} is invalid")

In [15]:
def gradient_descent(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_va: np.ndarray,
    y_va: np.ndarray,
    loss_fn: str,
    lambda_: float,
    batch_size: int = -1,
    beta: float = 0.9,
    gamma: float = 0.1,
    n_iters: int = 10000,
    valid_every: int = 10,
):
    n_tr, D = X_tr.shape
    pos_ids = np.where(y_tr[:, 0] == 1)[0]
    neg_ids = np.where(y_tr[:, 0] == -1)[0]

    w = np.random.randn(D, 1) / np.sqrt(D)

    best_w = w.copy()
    best_id = None
    loss_list = []
    val_loss_list = []
    acc_list = []
    f1_list = []
    loss_list.append(compute_loss(X_tr, y_tr, w, loss_fn, lambda_))
    val_loss_list.append(compute_loss(X_va, y_va, w, loss_fn, lambda_))
    acc_list.append(acc(X_va, y_va, w))
    f1_list.append(f1(X_va, y_va, w))
    momentum = np.zeros_like(w)
    with tqdm(range(n_iters), bar_format="{l_bar}{bar:10}{r_bar}") as pbar:
        for i in pbar:
            if batch_size > 0:
                n_pos = batch_size // 2
                batch_ids = np.concatenate(
                    [np.random.choice(pos_ids, n_pos, replace=True), 
                     np.random.choice(neg_ids, batch_size - n_pos, replace=True)],
                    axis=0,
                )
            else:
                batch_ids = np.arange(n_tr)
            
            grad = compute_gradient(X_tr[batch_ids], y_tr[batch_ids], w, loss_fn, lambda_)
            
            if i == 0:
                momentum = grad
            else:
                momentum = beta * momentum + grad
                
            w = w - gamma * momentum
            
            if (i + 1) % valid_every == 0:
                loss_list.append(compute_loss(X_tr, y_tr, w, loss_fn, lambda_))
                val_loss_list.append(compute_loss(X_va, y_va, w, loss_fn, lambda_))
                acc_list.append(acc(X_va, y_va, w))
                f1_list.append(f1(X_va, y_va, w))

                if best_id is None or f1_list[(best_id + 1) // valid_every] < f1_list[-1]:
                    best_id = i
                    best_w = w.copy()
                
                pbar.set_postfix({
                    "loss": loss_list[-1],
                    "val_loss": val_loss_list[-1],
                    "acc": acc_list[-1],
                    "f1": f1_list[-1]
                })
            
    return w, best_id, best_w, loss_list, val_loss_list, acc_list, f1_list

In [16]:
def one_test(x_np, y_tr_np, n_trains, loss_fn: str, lambda_: float = 0.1, gamma: float = 1e-5):
    (X_tr, y_tr), (X_va, y_va) = split_data(x_np, y_tr_np, 0.8)
    
    w, best_id, best_w, loss_list, val_loss_list, acc_list, f1_list = gradient_descent(X_tr, y_tr, X_va, y_va, 
                                            loss_fn=loss_fn, lambda_=lambda_, batch_size=2 ** 10, 
                                            beta=0.9, gamma=gamma, n_iters=32000, valid_every=1000)
    fig = plt.figure()
    ax = fig.add_subplot()
    name = f"save_data/{x_np.shape}_{loss_fn}_{lambda_}_{gamma}"
    ax.set_title(name)
    ax.plot(np.arange(0, len(loss_list)), loss_list)
    ax.plot(np.arange(0, len(loss_list)), val_loss_list)
    fig.savefig(f"{name}.pdf")
    np.save(f"{name}.npy", best_w)
    return best_w, acc(X_va, y_va, best_w), f1(X_va, y_va, best_w)

In [ ]:
best_w = None
best_f1 = None
best_n_degrees = None
for cat_t in [5, 10]:
    for n_degrees in [3, 7]:
        CAT_T = cat_t
        N_DEGREES = n_degrees
        
        new_cols = [np.ones(x_np.shape[0])]
        for i in range(1, x_np.shape[-1]):
            col = x_np[:, i]
            u_values = np.unique(col)
            if u_values.shape[0] <= CAT_T:
                for v in u_values:
                    if np.isnan(v):
                        new_cols.append(np.isnan(col).astype(np.float64))
                    else:
                        new_cols.append((col == v).astype(np.float64))
            else:
                mean = np.mean(col[~np.isnan(col)])
                std = np.std(col[~np.isnan(col)])
                norm_col = np.nan_to_num((col - mean) / std, 0)
                for d in range(1, N_DEGREES + 1):
                    new_cols.append(norm_col ** d)
                    
        
        pp_x_np = np.stack(new_cols, axis=-1)
        pp_x_tr_np = pp_x_np[:n_trains]
        
        for loss_fn in ["hinge", "logistic"]:
            for lambda_ in [1e-6, 1e-4, 1e-2, 1]:
                for gamma in [1e-5, 1e-6]:
                    print("##")
                    print("\tcat_t =", cat_t)
                    print("\tn_degree =", n_degrees)
                    print("\tloss_fn =", loss_fn)
                    print("\tlambda_ =", lambda_)
                    print("\tgamma =", gamma)
                    w, va_acc, va_f1 = one_test(
                        pp_x_tr_np, y_tr_np[:, 1:], n_trains, 
                        loss_fn=loss_fn, 
                        lambda_=lambda_, 
                        gamma=gamma, 
                    )
                    print("## acc =", va_acc)
                    print("## f1 =", va_f1)
                    if best_f1 is None or best_f1 < va_f1:
                        best_f1 = va_f1
                        best_w = w.copy()
                        best_n_degrees = n_degrees
                        print("## New best")
                    
np.save(f"best_w_d{best_n_degrees}.npy", best_w)

##
	cat_t = 5
	n_degree = 3
	loss_fn = hinge
	lambda_ = 1e-06
	gamma = 1e-05


100%|██████████| 32000/32000 [02:27<00:00, 216.91it/s, loss=0.895, val_loss=1.73, acc=0.71, f1=0.323] 


## acc = 0.7268045164337849
## f1 = 0.3346075304887607
## New best
##
	cat_t = 5
	n_degree = 3
	loss_fn = hinge
	lambda_ = 1e-06
	gamma = 1e-06


100%|██████████| 32000/32000 [02:30<00:00, 213.13it/s, loss=0.723, val_loss=0.808, acc=0.692, f1=0.311]


## acc = 0.692397945967361
## f1 = 0.3109533367586788
##
	cat_t = 5
	n_degree = 3
	loss_fn = hinge
	lambda_ = 0.0001
	gamma = 1e-05


...

## acc = 0.6874304783092324
## f1 = 0.29896448880924975
##
	cat_t = 5
	n_degree = 3
	loss_fn = logistic
	lambda_ = 1
	gamma = 1e-05


100%|██████████| 32000/32000 [01:15<00:00, 426.37it/s, loss=0.868, val_loss=1.52, acc=0.701, f1=0.31]  


## acc = 0.7059137245341094
## f1 = 0.3213783371205916
##
	cat_t = 5
	n_degree = 3
	loss_fn = logistic
	lambda_ = 1
	gamma = 1e-06


100%|██████████| 32000/32000 [01:14<00:00, 430.87it/s, loss=0.772, val_loss=1.8, acc=0.689, f1=0.302]  


## acc = 0.6887256769317507
## f1 = 0.30170232812787634
